# OpenComp_FoM: Reproducible Dynamic Comparator Co-Optimization in SKY130

**Track:** IEEE SSCS TC-OSE Code-a-Chip (VLSI'26)  
**Topologies compared:** StrongARM vs Double-Tail  
**Primary objective:** multi-objective optimization of delay, energy, and input-referred noise under PVT + mismatch

## Team
- Team Lead: William Anthony
- Members: Benedictus Kenneth Setiadi

## License
This notebook is intended for release under **Apache-2.0** (add LICENSE file in the project folder).

## Why this notebook
This notebook delivers a reproducible, educational, and design-relevant exploration of dynamic comparator trade-offs:
- Circuit-level comparison of two practical topologies.
- Unified score for decision making:
  $$S = w_1 \cdot t_{delay,norm} + w_2 \cdot E_{norm} + w_3 \cdot \sigma_{V_{in,eq},norm}$$
- Full Pareto fronts (delay-energy-noise).
- Robustness ranking across PVT corners and Monte Carlo mismatch.
- Educational animation of regeneration dynamics.

## 0. Environment Setup (Colab-Compatible)
This section installs open-source layout/simulation tools used in notebook-driven chip design flows.
Run Cell 3 first. It may take several minutes.

If you are not on Colab/Linux, skip these shell cells and use your local pre-installed toolchain.

In [ ]:
# clone and install magic
%cd /content
!git clone https://github.com/RTimothyEdwards/magic.git
%cd magic
!./configure
!make
!make install

# clone and install netgen
%cd /content
!git clone https://github.com/RTimothyEdwards/netgen
%cd netgen
!./configure
!make
!make install

# clone and compile ngspice
%cd /content
!apt install bison flex libx11-dev libx11-6 libxaw7-dev libreadline6-dev autoconf libtool automake -y
!git clone https://git.code.sf.net/p/ngspice/ngspice
!cd ngspice && ./compile_linux.sh

# install openvaf
!wget https://openva.fra1.cdn.digitaloceanspaces.com/openvaf_23_5_0_linux_amd64.tar.gz
!tar -xf openvaf_23_5_0_linux_amd64.tar.gz

# install klayout standalone python package
!pip install klayout

### 0.1 Refresh PATH after install
If the runtime was reset, run the next cell to restore `/content` tools into `PATH`.

In [ ]:
# Optional: uncomment in a clean Colab/Jupyter runtime
# !pip install -q numpy pandas matplotlib seaborn scipy tqdm

import os
import json
import math
import shutil
import subprocess
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import animation
from tqdm.auto import tqdm

sns.set_context('talk')
sns.set_style('whitegrid')
np.random.seed(42)

print('Python environment ready')

In [ ]:
# Reproducibility configuration
PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
NETLIST_DIR = ARTIFACT_DIR / 'netlists'
RAW_DIR = ARTIFACT_DIR / 'raw'
PLOT_DIR = ARTIFACT_DIR / 'plots'
for d in [ARTIFACT_DIR, NETLIST_DIR, RAW_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEMO_MODE = True  # Set False to use ngspice real simulation mode
NGSPICE_BIN = shutil.which('ngspice') or 'ngspice'

WEIGHTS = {
    'delay': 0.40,
    'energy': 0.35,
    'sigma_vin_eq': 0.25,
}

TARGET_SPECS = {
    'delay_ps': 120.0,
    'energy_fJ': 12.0,
    'sigma_vin_eq_mV': 2.0,
}

PVT_CORNERS = [
    {'name': 'TT_1p8V_27C', 'vdd': 1.8, 'temp': 27, 'proc': 'tt'},
    {'name': 'SS_1p62V_125C', 'vdd': 1.62, 'temp': 125, 'proc': 'ss'},
    {'name': 'FF_1p98V_-40C', 'vdd': 1.98, 'temp': -40, 'proc': 'ff'},
]

MC_RUNS = 80
print('Configured artifacts, targets, PVT corners, and scoring weights')

In [ ]:
# execute this cell again after reseting the session
import os
PATH = os.environ['PATH']
%env PATH={PATH}:/content

### 0.2 Download PDK and compile models
This step pulls IHP Open PDK and compiles Verilog-A models with OpenVAF.
It also clones the reference automation repository used in the original DAC setup flow.

In [ ]:
%cd /home
# clone pdk
!git clone --recursive --branch dev https://github.com/IHP-GmbH/IHP-Open-PDK.git
# clone project repo
!git clone https://github.com/SDMote/Automated_DAC_Design
# compile models
%cd /home/IHP-Open-PDK/ihp-sg13g2/libs.tech/verilog-a
!./openvaf-compile-va.sh

## Design Variables and Search Space
The same abstraction is used for both topologies. You can tighten ranges for runtime control.

In [ ]:
@dataclass
class DesignPoint:
    topology: str
    wn_um: float
    wp_um: float
    l_um: float
    itail_uA: float
    cl_fF: float

def build_design_space():
    points = []
    topology_list = ['StrongARM', 'DoubleTail']

    wn_vals = [0.6, 0.9, 1.2, 1.6, 2.0]
    wp_vals = [0.8, 1.2, 1.6, 2.2]
    l_vals = [0.15, 0.18, 0.22]
    itail_vals = [20, 35, 50, 70, 95]
    cl_vals = [2, 5, 10]

    for t in topology_list:
        for wn in wn_vals:
            for wp in wp_vals:
                for l in l_vals:
                    for it in itail_vals:
                        for cl in cl_vals:
                            points.append(DesignPoint(t, wn, wp, l, it, cl))
    return points

design_space = build_design_space()
print(f'Design points: {len(design_space)}')

## Netlist Generation (Template Hooks)
Replace model includes and node names with your validated schematic-level comparator netlists.

In [ ]:
def generate_netlist(point: DesignPoint, corner: dict, out_path: Path):
    # NOTE: Replace with your validated transistor-level netlist includes and measurements.
    model_card = {
        'tt': 'sky130_tt.spice',
        'ss': 'sky130_ss.spice',
        'ff': 'sky130_ff.spice',
    }[corner['proc']]

    netlist = f"""
* Auto-generated comparator deck
.include {model_card}
.param VDD={corner['vdd']} TEMP={corner['temp']}
.param WN={point.wn_um}u WP={point.wp_um}u LCH={point.l_um}u
.param ITAIL={point.itail_uA}u CL={point.cl_fF}f

* TODO: instantiate {point.topology} comparator here
* XCOMP in_p in_n out_p out_n vdd vss comparator_{point.topology}

* Example stimulus (replace with your validated input ramp/pulse)
VinP in_p 0 PULSE(0 5m 1n 20p 20p 2n 10n)
VinN in_n 0 DC 0
VDD vdd 0 {corner['vdd']}
VSS vss 0 0
Cloadp out_p 0 {{CL}}
Cloadn out_n 0 {{CL}}

.tran 1p 12n

* TODO: adapt measure expressions to your node naming/sign convention
.measure tran t_delay trig v(in_p) val=2.5m rise=1 targ v(out_p) val='VDD/2' rise=1
.measure tran e_comp integ par('v(vdd)*i(VDD)') from=0n to=12n
.measure tran vmeta max v(out_p) from=0n to=12n

.end
"""
    out_path.write_text(netlist)

print('Netlist generator ready')

In [ ]:
def synthetic_metrics(point: DesignPoint, corner: dict, mc_sigma=0.0):
    # Surrogate model for reproducible demos when ngspice is unavailable.
    topo_factor = 1.0 if point.topology == 'StrongARM' else 0.92
    proc_delay = {'tt': 1.0, 'ss': 1.25, 'ff': 0.82}[corner['proc']]
    proc_noise = {'tt': 1.0, 'ss': 1.18, 'ff': 0.88}[corner['proc']]
    proc_energy = {'tt': 1.0, 'ss': 1.08, 'ff': 0.95}[corner['proc']]

    delay_ps = (
        260
        * topo_factor
        * proc_delay
        * (1.8 / corner['vdd'])
        * (point.cl_fF / 5.0) ** 0.45
        * (35.0 / point.itail_uA) ** 0.55
        * (0.18 / point.l_um) ** 0.20
    )

    energy_fJ = (
        8.0
        * (point.itail_uA / 35.0) ** 0.95
        * (corner['vdd'] / 1.8) ** 1.9
        * (point.cl_fF / 5.0) ** 0.30
        * proc_energy
    )

    sigma_vin_eq_mV = (
        3.2
        * proc_noise
        * (0.9 / point.wn_um) ** 0.40
        * (0.18 / point.l_um) ** 0.35
        * (35.0 / point.itail_uA) ** 0.15
    )

    if mc_sigma > 0:
        delay_ps *= np.random.lognormal(mean=0.0, sigma=mc_sigma)
        energy_fJ *= np.random.lognormal(mean=0.0, sigma=mc_sigma * 0.8)
        sigma_vin_eq_mV *= np.random.lognormal(mean=0.0, sigma=mc_sigma * 1.1)

    return {
        'delay_ps': float(delay_ps),
        'energy_fJ': float(energy_fJ),
        'sigma_vin_eq_mV': float(sigma_vin_eq_mV),
    }

def parse_measure_output(stdout_text: str):
    # Extend parser for your .measure formatting.
    metrics = {}
    for line in stdout_text.splitlines():
        low = line.lower()
        if 't_delay' in low and '=' in line:
            metrics['delay_ps'] = float(line.split('=')[-1].strip()) * 1e12
        if 'e_comp' in low and '=' in line:
            metrics['energy_fJ'] = float(line.split('=')[-1].strip()) * 1e15
    return metrics

def run_one(point: DesignPoint, corner: dict):
    if DEMO_MODE:
        return synthetic_metrics(point, corner, mc_sigma=0.0)

    deck = NETLIST_DIR / f"{point.topology}_{corner['name']}_{hash((point.wn_um, point.wp_um, point.l_um, point.itail_uA, point.cl_fF))}.sp"
    generate_netlist(point, corner, deck)
    cmd = [NGSPICE_BIN, '-b', str(deck)]
    proc = subprocess.run(cmd, capture_output=True, text=True)

    if proc.returncode != 0:
        return {'delay_ps': np.nan, 'energy_fJ': np.nan, 'sigma_vin_eq_mV': np.nan}

    m = parse_measure_output(proc.stdout + '\n' + proc.stderr)
    if 'sigma_vin_eq_mV' not in m:
        # Placeholder until transient-noise extraction is wired to your deck
        m['sigma_vin_eq_mV'] = np.nan
    return m

print('Simulation runner ready (DEMO_MODE =', DEMO_MODE, ')')

In [ ]:
# Nominal + PVT sweep
records = []
for point in tqdm(design_space, desc='Sweeping design points'):
    for corner in PVT_CORNERS:
        m = run_one(point, corner)
        records.append({
            **asdict(point),
            'corner': corner['name'],
            'proc': corner['proc'],
            'vdd': corner['vdd'],
            'temp': corner['temp'],
            **m,
        })

df = pd.DataFrame.from_records(records)
display(df.head())
print('Rows:', len(df))

In [ ]:
# Monte Carlo robustness on shortlisted points (runtime control)
def shortlist(df_in: pd.DataFrame, per_topology=12):
    base = df_in[df_in['corner'] == 'TT_1p8V_27C'].copy()
    base = base.dropna(subset=['delay_ps', 'energy_fJ', 'sigma_vin_eq_mV'])
    if base.empty:
        return base

    for c in ['delay_ps', 'energy_fJ', 'sigma_vin_eq_mV']:
        x = base[c].values
        base[c + '_n'] = (x - x.min()) / (x.max() - x.min() + 1e-12)

    base['score_nom'] = (
        WEIGHTS['delay'] * base['delay_ps_n'] +
        WEIGHTS['energy'] * base['energy_fJ_n'] +
        WEIGHTS['sigma_vin_eq'] * base['sigma_vin_eq_mV_n']
    )

    rows = []
    for topo, g in base.groupby('topology'):
        rows.append(g.nsmallest(per_topology, 'score_nom'))
    return pd.concat(rows, ignore_index=True)

short_df = shortlist(df, per_topology=10)
print('Shortlisted for Monte Carlo:', len(short_df))
display(short_df[['topology','wn_um','wp_um','l_um','itail_uA','cl_fF','score_nom']].head())

In [ ]:
mc_records = []
for _, row in tqdm(short_df.iterrows(), total=len(short_df), desc='Monte Carlo'):
    point = DesignPoint(
        topology=row['topology'],
        wn_um=row['wn_um'],
        wp_um=row['wp_um'],
        l_um=row['l_um'],
        itail_uA=row['itail_uA'],
        cl_fF=row['cl_fF'],
    )

    for i in range(MC_RUNS):
        if DEMO_MODE:
            m = synthetic_metrics(point, {'proc':'tt','vdd':1.8,'temp':27,'name':'TT_1p8V_27C'}, mc_sigma=0.10)
        else:
            # TODO: replace with mismatch-enabled ngspice deck invocation
            m = run_one(point, {'proc':'tt','vdd':1.8,'temp':27,'name':'TT_1p8V_27C'})

        mc_records.append({
            **asdict(point),
            'mc_run': i,
            **m,
        })

df_mc = pd.DataFrame(mc_records)
display(df_mc.head())
print('MC rows:', len(df_mc))

## Unified Score + Robustness Ranking
Lower score is better after min-max normalization.

In [ ]:
def minmax_norm(s: pd.Series):
    return (s - s.min()) / (s.max() - s.min() + 1e-12)

# Aggregate PVT metrics by worst-case and mean
agg = (
    df.groupby(['topology','wn_um','wp_um','l_um','itail_uA','cl_fF'], as_index=False)
      .agg(
          delay_ps_mean=('delay_ps','mean'),
          delay_ps_worst=('delay_ps','max'),
          energy_fJ_mean=('energy_fJ','mean'),
          energy_fJ_worst=('energy_fJ','max'),
          sigma_vin_eq_mV_mean=('sigma_vin_eq_mV','mean'),
          sigma_vin_eq_mV_worst=('sigma_vin_eq_mV','max'),
      )
)

mc_stat = (
    df_mc.groupby(['topology','wn_um','wp_um','l_um','itail_uA','cl_fF'], as_index=False)
         .agg(
             delay_mc_p95=('delay_ps', lambda x: np.percentile(x, 95)),
             energy_mc_p95=('energy_fJ', lambda x: np.percentile(x, 95)),
             noise_mc_p95=('sigma_vin_eq_mV', lambda x: np.percentile(x, 95)),
         )
)

rank_df = agg.merge(mc_stat, on=['topology','wn_um','wp_um','l_um','itail_uA','cl_fF'], how='left')

rank_df['delay_n'] = minmax_norm(rank_df['delay_mc_p95'].fillna(rank_df['delay_ps_worst']))
rank_df['energy_n'] = minmax_norm(rank_df['energy_mc_p95'].fillna(rank_df['energy_fJ_worst']))
rank_df['noise_n'] = minmax_norm(rank_df['noise_mc_p95'].fillna(rank_df['sigma_vin_eq_mV_worst']))

rank_df['score_unified'] = (
    WEIGHTS['delay'] * rank_df['delay_n'] +
    WEIGHTS['energy'] * rank_df['energy_n'] +
    WEIGHTS['sigma_vin_eq'] * rank_df['noise_n']
)

rank_df = rank_df.sort_values('score_unified', ascending=True).reset_index(drop=True)
display(rank_df.head(10))

In [ ]:
# Pareto frontier (delay, energy, noise)
def pareto_front(df_in, cols=('delay_ps_worst','energy_fJ_worst','sigma_vin_eq_mV_worst')):
    vals = df_in.loc[:, cols].values
    is_efficient = np.ones(vals.shape[0], dtype=bool)

    for i, c in enumerate(vals):
        if is_efficient[i]:
            is_efficient[is_efficient] = np.any(vals[is_efficient] < c, axis=1) | np.all(vals[is_efficient] == c, axis=1)
            is_efficient[i] = True
    return df_in[is_efficient].copy()

pareto_df = pd.concat([pareto_front(g) for _, g in rank_df.groupby('topology')], ignore_index=True)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
sns.scatterplot(data=rank_df, x='delay_ps_worst', y='energy_fJ_worst', hue='topology', alpha=0.25, ax=ax[0])
sns.scatterplot(data=pareto_df, x='delay_ps_worst', y='energy_fJ_worst', hue='topology', s=120, marker='X', ax=ax[0], legend=False)
ax[0].set_title('Delay-Energy Pareto (worst-case PVT)')

sns.scatterplot(data=rank_df, x='delay_ps_worst', y='sigma_vin_eq_mV_worst', hue='topology', alpha=0.25, ax=ax[1])
sns.scatterplot(data=pareto_df, x='delay_ps_worst', y='sigma_vin_eq_mV_worst', hue='topology', s=120, marker='X', ax=ax[1], legend=False)
ax[1].set_title('Delay-Noise Pareto (worst-case PVT)')

plt.tight_layout()
plt.savefig(PLOT_DIR / 'pareto_fronts.png', dpi=180)
plt.show()

## Educational Animation: Regeneration Dynamics
This section visualizes metastability escape and differential output growth for representative design points.

In [ ]:
# Select representative points for animation
best_overall = rank_df.iloc[0]
best_strongarm = rank_df[rank_df['topology'] == 'StrongARM'].iloc[0]
best_doubletail = rank_df[rank_df['topology'] == 'DoubleTail'].iloc[0]

reps = pd.DataFrame([best_overall, best_strongarm, best_doubletail]).drop_duplicates()
display(reps[['topology','wn_um','wp_um','l_um','itail_uA','cl_fF','score_unified']])

t = np.linspace(0, 1.2e-9, 240)

def regen_waveform(row, tvec):
    # Educational surrogate: sigmoid growth linked to delay and current
    tau = max(8e-11, row['delay_ps_worst'] * 1e-12 / 3.0)
    amp = 0.9 if row['topology'] == 'DoubleTail' else 1.0
    v = amp * (1 / (1 + np.exp(-(tvec - 4*tau)/tau)) - 0.5)
    return v

waves = [regen_waveform(r, t) for _, r in reps.iterrows()]
labels = [f"{r['topology']} (score={r['score_unified']:.3f})" for _, r in reps.iterrows()]

fig, ax = plt.subplots(figsize=(10, 5))
lines = [ax.plot([], [], lw=2, label=lab)[0] for lab in labels]
ax.set_xlim(t.min()*1e9, t.max()*1e9)
ax.set_ylim(-0.6, 0.6)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Differential Output (a.u.)')
ax.set_title('Comparator Regeneration Dynamics (Educational)')
ax.legend(loc='lower right')

def init():
    for ln in lines:
        ln.set_data([], [])
    return lines

def animate(i):
    x = t[:i] * 1e9
    for ln, w in zip(lines, waves):
        ln.set_data(x, w[:i])
    return lines

ani = animation.FuncAnimation(fig, animate, frames=len(t), init_func=init, interval=24, blit=True)
gif_path = PLOT_DIR / 'regeneration_dynamics.gif'
ani.save(gif_path, writer='pillow', fps=30)
plt.close(fig)
print('Saved animation:', gif_path)

## Spec-to-Sizing Recipe (5 Steps)
This deterministic recipe converts target specs into a recommended starting design point.

In [ ]:
def recommend_from_specs(rank_table: pd.DataFrame, specs: dict):
    tmp = rank_table.copy()
    tmp['pass_delay'] = tmp['delay_ps_worst'] <= specs['delay_ps']
    tmp['pass_energy'] = tmp['energy_fJ_worst'] <= specs['energy_fJ']
    tmp['pass_noise'] = tmp['sigma_vin_eq_mV_worst'] <= specs['sigma_vin_eq_mV']
    tmp['num_pass'] = tmp[['pass_delay','pass_energy','pass_noise']].sum(axis=1)

    feasible = tmp[tmp['num_pass'] == 3]
    if not feasible.empty:
        pick = feasible.nsmallest(1, 'score_unified').iloc[0]
        reason = 'All specs satisfied; selected minimum unified score among feasible points.'
    else:
        pick = tmp.sort_values(['num_pass','score_unified'], ascending=[False, True]).iloc[0]
        reason = 'No fully feasible point; selected best trade-off by pass count then score.'

    return pick, reason

best_pick, why = recommend_from_specs(rank_df, TARGET_SPECS)
display(pd.DataFrame([best_pick])[['topology','wn_um','wp_um','l_um','itail_uA','cl_fF','delay_ps_worst','energy_fJ_worst','sigma_vin_eq_mV_worst','score_unified']])
print(why)

### Recipe Narrative
1. Set system-level budgets for delay, energy, and input-referred noise.  
2. Sweep both topologies over a common transistor/load/current parameter space.  
3. Rank points by PVT worst-case and MC p95 using the unified score.  
4. Filter by hard constraints, then choose the minimum score candidate. 
5. Validate chosen point with full transistor-level decks and signoff-oriented sweeps.

In [ ]:
# Save reproducible artifacts
df.to_csv(ARTIFACT_DIR / 'sweep_pvt.csv', index=False)
df_mc.to_csv(ARTIFACT_DIR / 'sweep_mc.csv', index=False)
rank_df.to_csv(ARTIFACT_DIR / 'ranking.csv', index=False)
pareto_df.to_csv(ARTIFACT_DIR / 'pareto.csv', index=False)

summary = {
    'demo_mode': DEMO_MODE,
    'weights': WEIGHTS,
    'target_specs': TARGET_SPECS,
    'best_point': {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in best_pick.to_dict().items()},
}
(ARTIFACT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))
print('Artifacts exported to:', ARTIFACT_DIR)

## References
- Code-a-Chip main page and rules.
- Comparator theory and relevant prior publications.
- Open-source PDK/SPICE documentation used for this notebook.